# Notebook 11: Food photos

**Goal:** give every recipe card a good-looking photo.

**Why:** only 35% of the catalogue has a Food.com photo, and many of those are low quality.

**Agreed decisions (17 Nov):**
1. Photos come from **Unsplash** (free API).
2. There is **no photo of each exact recipe**. Instead, every recipe gets a photo of its **dish type** (e.g. *banana bread*, *lentil soup*, *chicken curry*), found from the recipe name with an ordered list of rules (the first match wins).
3. Recipes with no matching rule, and rules used by fewer than 5 recipes, fall back to a photo for their **meal type** (e.g. *dessert*, *side dish*).
4. Up to **5 photos per dish type**, so similar recipes do not all look the same. A recipe always gets the same photo (chosen from its id).
5. The app shows the photo as an **illustration** and credits the photographer and Unsplash (Unsplash API guidelines).
6. This runs **once**. The app only reads the saved links, so it needs **no API key**.

**Unsplash rules we follow:** photos are hotlinked from Unsplash's own URLs (not downloaded), and every photo shows "Photo by *name* on Unsplash" with links.

**Rate limit:** a new (demo) Unsplash app allows **50 requests per hour**. We need about 140 (one per dish type), so the fetch takes about 3 hours. It saves after every request and waits by itself when the limit is reached. If you stop it, run the cell again and it continues where it stopped.

**Outputs (`../app/data/`):**

| file | what |
|---|---|
| `photo_rules.csv` | ordered rules: dish type (query) → regex on the recipe name |
| `photo_fallback.csv` | meal type → fallback query |
| `photos.csv` | up to 5 photos per query: Unsplash id, image URL, photographer name and links |

In [1]:
import json
import re
import time
import urllib.parse
import urllib.request
import urllib.error
from datetime import datetime
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

DATA_DIR = "../app/data"                       # path strings
RECIPES_FILE = DATA_DIR + "/recipes_app.parquet"
CACHE_FILE = "unsplash_cache.json"             # raw search results, so a stopped run can continue
RULES_FILE = DATA_DIR + "/photo_rules.csv"
FALLBACK_FILE = DATA_DIR + "/photo_fallback.csv"
PHOTOS_FILE = DATA_DIR + "/photos.csv"

MIN_RECIPES = 5      # a dish type needs at least this many recipes to get its own photos
N_PHOTOS = 5         # photos kept per dish type
PER_PAGE = 10        # results asked per search (we keep the best N_PHOTOS not used elsewhere)

## 1. Recipes

In [2]:
recipes = pd.read_parquet(RECIPES_FILE, columns=["RecipeId", "Name", "meal_type"])
print(recipes.shape)
recipes["meal_type"].value_counts()

(12375, 3)


meal_type
Desserts & baking      6319
Salads & sides         1856
Main                   1797
Breakfast               794
Snacks & appetizers     712
Other                   477
Sauces & dressings      420
Name: count, dtype: int64

## 2. Dish-type rules

Each rule is *(search query, regex on the recipe name)*. **Order matters:** specific dishes come before general ones (e.g. *banana bread* before *bread loaf*, *meatloaf* before *bread loaf*, *cheesecake* before *cake slice*).

In [3]:
# (search query for Unsplash, regex on the recipe name). First match wins, so specific dishes come first.
DISH_RULES = [
    # breakfast
    ("pancakes", r"pancake"), ("waffles", r"waffle"), ("crepes", r"crepe|crêpe"), ("french toast", r"french toast"),
    ("omelette", r"omelet|frittata"), ("quiche", r"quiche"), ("scrambled eggs", r"scrambl"), ("eggs breakfast", r"\beggs?\b|benedict"),
    ("oatmeal bowl", r"^(?!.*(cookie|muffin|bread|bar|cake|pie|crisp|pancake|scone|loaf)).*(oatmeal|porridge|overnight oats)"), ("granola", r"granola|muesli"), ("smoothie", r"smoothie|shake\b"),
    # baking & desserts
    ("cheesecake", r"cheesecake"), ("cupcakes", r"cupcake"), ("chocolate cake", r"chocolate.*cake|cake.*chocolate|devil'?s food"),
    ("carrot cake", r"carrot cake"), ("coffee cake", r"coffee ?cake|streusel"), ("pound cake", r"pound cake|bundt"),
    ("banana bread", r"banana.*(bread|loaf)"), ("zucchini bread", r"zucchini (bread|loaf)|pumpkin (bread|loaf)"),
    ("cake slice", r"\bcakes?\b|torte|gateau"),
    ("brownies", r"brownie|blondie"), ("chocolate chip cookies", r"chocolate chip cookie|choc chip"),
    ("cookies", r"cookie|biscotti|snickerdoodle|shortbread|macaroon|gingersnap|\bsnaps\b"), ("bar dessert", r"\bbars?\b|squares"),
    ("apple pie", r"apple (pie|crisp|crumble|cobbler)"), ("pumpkin pie", r"pumpkin pie|sweet potato pie"), ("pecan pie", r"pecan pie"),
    ("berry pie", r"(berry|cherry|peach).*(pie|cobbler|crisp|crumble|tart)"), ("pie", r"\bpies?\b|tart\b|tarts\b|tartlet|cobbler|crisp\b|crumble|galette"),
    ("muffins", r"muffin"), ("donuts", r"donut|doughnut"), ("cinnamon rolls", r"cinnamon roll|sticky bun|sweet roll"),
    ("chocolate fudge", r"fudge|truffle|bonbon"), ("candy", r"candy|toffee|brittle|caramel|marshmallow|lollipop|bark\b"),
    ("pudding dessert", r"pudding|custard|flan|mousse|panna cotta|creme brulee|trifle|parfait"),
    ("ice cream", r"ice cream|sorbet|gelato|frozen yogurt|popsicle|semifreddo"),
    ("meatloaf", r"meatloaf|meat loaf"), ("scones", r"scone"), ("biscuits bread", r"biscuit"), ("cornbread", r"cornbread|corn bread|johnny cake"),
    ("bagels", r"bagel|pretzel"), ("dinner rolls", r"\brolls?\b|\bbuns?\b|brioche"), ("focaccia", r"focaccia|flatbread|naan|pita\b"),
    ("bread loaf", r"bread|loaf|sourdough|baguette|challah"),
    # world dishes & mains
    ("pizza", r"pizza|calzone"), ("burger", r"burger|slider"), ("tacos", r"taco"), ("burrito", r"burrito|wrap\b|wraps\b"),
    ("enchiladas", r"enchilada|quesadilla|fajita|tostada|tamale"), ("nachos", r"nacho"), ("sushi", r"sushi|maki\b"),
    ("ramen noodles", r"ramen|noodle|pho\b|lo mein|pad thai|udon|chow mein"), ("dumplings", r"dumpling|wonton|pierogi|potsticker|gyoza"),
    ("samosa", r"samosa|empanada|turnover|hand pie"), ("lasagna", r"lasagn"), ("mac and cheese", r"mac.*cheese|macaroni"),
    ("spaghetti", r"spaghetti|bolognese|meatballs? and"), ("pasta dish", r"pasta|penne|fettuccin|linguin|ravioli|tortellini|gnocchi|orzo|rigatoni|ziti|manicotti|carbonara|alfredo"),
    ("risotto", r"risotto"), ("paella", r"paella|jambalaya"), ("fried rice", r"fried rice"), ("curry", r"curry|masala|korma|tikka|vindaloo"),
    ("dal lentils", r"\bdal\b|dhal|daal"), ("stir fry", r"stir.?fry|teriyaki|szechuan|kung pao|general tso|sweet and sour"),
    ("chili con carne", r"\bchili\b|chilli"), ("lentil soup", r"lentil soup"), ("tomato soup", r"tomato soup|gazpacho"),
    ("chicken soup", r"chicken.*soup|chicken noodle"), ("chowder", r"chowder|bisque"), ("stew", r"stew|goulash|gumbo|tagine|ragout"),
    ("soup bowl", r"soup|broth|minestrone|borscht"), ("casserole", r"casserole|bake\b|gratin|hotdish|pot pie|shepherd"),
    ("meatballs", r"meatball"), ("sandwich", r"sandwich|panini|sub\b|\bmelts?\b|sloppy joe|po'? ?boy"),
    ("kebab", r"kebab|kabob|skewer|souvlaki|shawarma|gyro"), ("falafel hummus", r"falafel|hummus"),
    ("guacamole", r"guacamole|avocado dip"), ("salsa", r"salsa|pico de gallo"), ("dip platter", r"\bdip\b|dips\b|fondue|spread"),
    ("deviled eggs", r"deviled"), ("bruschetta", r"bruschetta|crostini|canape|toast\b"),
    # salads
    ("caesar salad", r"caesar"), ("pasta salad", r"pasta salad|macaroni salad"), ("potato salad", r"potato salad"),
    ("coleslaw", r"slaw"), ("fruit salad", r"fruit salad|ambrosia"), ("chicken salad", r"chicken salad|tuna salad|egg salad"),
    ("quinoa salad", r"quinoa|couscous|tabbouleh|bulgur"), ("bean salad", r"bean salad|chickpea salad|lentil salad"),
    ("green salad", r"salad"),
    # proteins
    ("shrimp dish", r"shrimp|prawn|scampi"), ("salmon fillet", r"salmon"), ("seafood", r"scallop|lobster|crab|mussel|clam|oyster|calamari|squid"),
    ("fish fillet", r"fish|tuna|cod\b|tilapia|halibut|trout|mahi|sole\b|haddock|catfish|snapper|sardine|swordfish"),
    ("chicken wings", r"wings?\b|drumstick"), ("roast chicken", r"roast(ed)? chicken|whole chicken|rotisserie"),
    ("turkey dinner", r"turkey"), ("grilled chicken", r"chicken|poultry|hen\b"),
    ("steak", r"steak|sirloin|tenderloin|filet"), ("pulled pork", r"pulled pork|carnitas"), ("pork chops", r"pork chop|pork loin|pork tenderloin"),
    ("ribs", r"\bribs?\b"), ("bacon", r"bacon"), ("sausage", r"sausage|chorizo|bratwurst|hot dog|kielbasa"), ("ham", r"\bham\b"),
    ("roast pork", r"pork"), ("lamb", r"lamb|mutton"), ("beef stew", r"pot roast|brisket|roast beef"), ("ground beef", r"beef|veal|venison|bison"),
    ("tofu", r"tofu|tempeh|seitan"),
    # sides & vegetables
    ("sweet potatoes", r"sweet potato|\byams?\b"), ("mashed potatoes", r"mashed potato|potato mash"), ("roasted potatoes", r"potato|fries|hash brown|tater"),
    ("rice dish", r"rice|pilaf|biryani"), ("bean dish", r"bean|lentil|chickpea|garbanzo"),
    ("roasted vegetables", r"roasted (veg|root)|vegetable|veggie"), ("broccoli", r"broccoli|cauliflower"), ("carrots", r"carrot"),
    ("asparagus", r"asparagus"), ("green beans", r"green bean|string bean|peas\b|snow pea|snap pea"), ("spinach", r"spinach|kale|chard|collard|greens"),
    ("mushrooms", r"mushroom"), ("zucchini", r"zucchini|squash|eggplant|aubergine|courgette"), ("stuffed peppers", r"stuffed pepper|bell pepper"),
    ("corn on the cob", r"\bcorn\b|polenta|grits"), ("pumpkin", r"pumpkin"), ("tomatoes", r"tomato"),
    ("cabbage", r"cabbage|sauerkraut|brussels"),
    # sauces, drinks & snacks
    ("salad dressing", r"dressing|vinaigrette"), ("pesto", r"pesto"), ("tomato sauce", r"marinara|tomato sauce|pasta sauce|spaghetti sauce"),
    ("bbq sauce", r"bbq|barbecue|marinade|rub\b"), ("jam jar", r"jam\b|jelly|preserves|marmalade|chutney|relish|compote|curd\b"),
    ("sauce bowl", r"sauce|gravy|glaze|syrup|aioli|mayonnaise|ketchup|butter\b"), ("pickles", r"pickle"),
    ("lemonade", r"lemonade|punch|limeade|agua fresca|spritzer"), ("cocktail", r"cocktail|martini|margarita|mojito|sangria|daiquiri|liqueur|eggnog"),
    ("hot chocolate", r"hot chocolate|cocoa"), ("coffee", r"coffee|latte|mocha|cappuccino|espresso|frappe"), ("tea", r"\btea\b|chai"),
    ("popcorn", r"popcorn"), ("trail mix", r"trail mix|snack mix|chex|granola bar|energy bites|energy balls"),
    ("nuts snack", r"almond|walnut|pecan|cashew|peanut|\bnuts?\b|pistachio"), ("crackers", r"cracker|chips\b|crisps"),
    ("cheese platter", r"cheese"), ("chocolate", r"chocolate"), ("banana", r"banana"), ("apples", r"\bapples?\b"),
    ("berries", r"berr|cherr"), ("lemon dessert", r"lemon|lime|orange|citrus"), ("peaches", r"peach|apricot|plum|nectarine|mango|pineapple"),
]
MEAL_FALLBACK = {"Main": "dinner plate", "Salads & sides": "side dish", "Desserts & baking": "dessert",
                 "Breakfast": "breakfast table", "Snacks & appetizers": "appetizers", "Sauces & dressings": "sauce",
                 "Drinks": "drink glass", "Other": "home cooking"}

print(len(DISH_RULES), "rules,", len(MEAL_FALLBACK), "meal-type fallbacks")

151 rules, 8 meal-type fallbacks


## 3. Give every recipe a dish type

In [4]:
compiled = [(q, re.compile(p, re.I)) for q, p in DISH_RULES]

def first_rule(name):
    for q, rx in compiled:
        if rx.search(str(name)):
            return q
    return None

recipes["rule_query"] = recipes["Name"].map(first_rule)
counts = recipes["rule_query"].value_counts()
small = set(counts[counts < MIN_RECIPES].index)          # too few recipes: use the meal-type photo instead

recipes["query"] = [q if (pd.notna(q) and q not in small) else MEAL_FALLBACK.get(m, "home cooking")
                    for q, m in zip(recipes["rule_query"], recipes["meal_type"])]

fallback_share = recipes["query"].isin(set(MEAL_FALLBACK.values())).mean()
print(f"dish types used: {recipes['query'].nunique()} | recipes on a meal-type fallback: {fallback_share:.1%}")
print("rules never matched:", [q for q, _ in DISH_RULES if q not in counts.index])
recipes["query"].value_counts().head(20)

dish types used: 137 | recipes on a meal-type fallback: 5.1%
rules never matched: ['sushi', 'paella', 'pasta salad', 'pulled pork', 'pork chops', 'bacon', 'sausage', 'ham', 'roast pork', 'beef stew', 'trail mix']


query
cake slice          836
cookies             693
grilled chicken     540
muffins             534
bread loaf          437
pie                 366
bar dessert         325
pudding dessert     273
pasta dish          243
dessert             242
banana bread        239
cheesecake          234
sauce bowl          231
soup bowl           229
green salad         220
coffee cake         220
casserole           214
rice dish           212
roasted potatoes    212
biscuits bread      182
Name: count, dtype: int64

Spot check: random recipes and their dish type.

In [5]:
recipes.sample(20, random_state=11)[["Name", "meal_type", "query"]]

,Name,meal_type,query
11149,Whole Wheat Butterscotch Brownies,Desserts & baking,brownies
2724,Paprika Chicken,Main,grilled chicken
9466,Healthy Nutella Rice,Snacks & appetizers,rice dish
219,Salmon Pizza,Snacks & appetizers,pizza
6463,Herbed Roasted Potatoes,Salads & sides,roasted potatoes
7812,Fruit Cake - Dump Cake,Desserts & baking,cake slice
4392,Best Ever White Bread (Abm),Desserts & baking,bread loaf
12116,Banana Spice Coffeecake,Desserts & baking,coffee cake
4110,Pancakes,Breakfast,pancakes
1728,Summer Strawberry Rhubarb Crisp,Desserts & baking,berry pie


## 4. Save the rules for the app

The app applies the same rules to card names, including the user's own recipes. If the matched dish type has no photos (a small or failed query), it uses the meal-type fallback.

In [6]:
rules_df = pd.DataFrame(DISH_RULES, columns=["query", "pattern"])
rules_df["order"] = range(len(rules_df))
rules_df[["order", "query", "pattern"]].to_csv(RULES_FILE, index=False)
pd.DataFrame(list(MEAL_FALLBACK.items()), columns=["meal_type", "query"]).to_csv(FALLBACK_FILE, index=False)
print("saved", RULES_FILE, "and", FALLBACK_FILE)

saved ../app/data/photo_rules.csv and ../app/data/photo_fallback.csv


## 5. Unsplash API key

Paste your **Access Key** when asked (it is hidden and not saved in the notebook). Never commit it to GitHub.

In [8]:
import getpass
ACCESS_KEY = getpass.getpass("Unsplash Access Key: ").strip()
print("key length:", len(ACCESS_KEY))

Unsplash Access Key:  ········


key length: 43


## 6. Search Unsplash (one request per dish type)

- Queries run from the most used dish type to the least, so the most visible cards are done first.
- Results are saved to `unsplash_cache.json` after every request.
- When the hourly limit is reached, the cell waits and tries again every 10 minutes. You can leave it running.

In [12]:
API = "https://api.unsplash.com/search/photos"
queries = recipes["query"].value_counts().index.tolist()      # most used first
cache = json.loads(Path(CACHE_FILE).read_text()) if Path(CACHE_FILE).exists() else {}
print(f"{len(queries)} queries, {sum(q in cache for q in queries)} already in the cache")


def search_unsplash(query):
    params = {"query": query, "per_page": PER_PAGE, "orientation": "landscape", "content_filter": "high"}
    req = urllib.request.Request(API + "?" + urllib.parse.urlencode(params),
                                 headers={"Authorization": f"Client-ID {ACCESS_KEY}", "Accept-Version": "v1"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        remaining = resp.headers.get("X-Ratelimit-Remaining")
        data = json.loads(resp.read().decode("utf-8"))
    photos = [{"photo_id": r["id"], "url_raw": r["urls"]["raw"], "alt": r.get("alt_description") or "",
               "author": r["user"]["name"], "author_url": r["user"]["links"]["html"], "photo_page": r["links"]["html"]}
              for r in data.get("results", [])]
    return photos, remaining


for i, q in enumerate(queries, 1):
    if q in cache:
        continue
    while True:
        try:
            photos, remaining = search_unsplash(q)
            break
        except urllib.error.HTTPError as e:
            if e.code in (403, 429):        # hourly limit reached
                print(f"{datetime.now():%H:%M} limit reached, waiting 10 minutes ...")
                time.sleep(600)
            elif e.code == 401:
                raise RuntimeError("The Access Key was not accepted. Run the key cell again.") from e
            else:
                raise
    cache[q] = photos
    Path(CACHE_FILE).write_text(json.dumps(cache))
    print(f"{i:>3}/{len(queries)}  {q:<25} {len(photos)} photos  (requests left this hour: {remaining})")
    time.sleep(1)

print("done:", sum(q in cache for q in queries), "of", len(queries))

137 queries, 137 already in the cache
done: 137 of 137


## 7. Pick the photos

Up to 5 per dish type, in Unsplash's relevance order. A photo already used by a more common dish type is skipped, so the same photo never shows for two different dishes.

In [13]:
rows, used = [], set()
for q in queries:
    kept = 0
    for p in cache.get(q, []):
        if p["photo_id"] in used:
            continue
        used.add(p["photo_id"])
        rows.append({"query": q, "rank": kept, **p})
        kept += 1
        if kept == N_PHOTOS:
            break

photos = pd.DataFrame(rows)
photos.to_csv(PHOTOS_FILE, index=False)
missing = [q for q in queries if q not in set(photos["query"])]
print(f"saved {len(photos)} photos for {photos['query'].nunique()} dish types -> {PHOTOS_FILE}")
print("dish types without photos (they use the meal-type photo):", missing)

saved 685 photos for 137 dish types -> ../app/data/photos.csv
dish types without photos (they use the meal-type photo): []


## 8. Visual check

In [14]:
def thumb(url_raw, w=220, h=150):
    return f"{url_raw}&w={w}&h={h}&fit=crop&auto=format&q=70"

html = ""
for q in queries[:24]:
    imgs = "".join(f'<img src="{thumb(u)}" style="margin:2px;border-radius:6px">'
                   for u in photos.loc[photos["query"] == q, "url_raw"].head(4))
    html += f"<div style='margin-bottom:8px'><b>{q}</b><br>{imgs}</div>"
display(HTML(html))

If a dish type shows wrong photos, change its query in section 2 (e.g. add a word), delete that query from `unsplash_cache.json` (or the whole file), and run sections 3 to 8 again. Only the changed queries are searched again.

## 9. Coverage

In [15]:
has_photos = set(photos["query"])
fallback_q = recipes["meal_type"].map(MEAL_FALLBACK).fillna("home cooking")
final_q = recipes["query"].where(recipes["query"].isin(has_photos), fallback_q)
print(f"recipes with a photo: {final_q.isin(has_photos).mean():.1%}")

recipes with a photo: 100.0%
